# 17. Generate predictions on the locked final test set


This notebook is the first stage that reads `final_test_rows_LOCKED.csv`. It loads the six frozen models, generates one probability per patient, and saves a single combined prediction table.

It does not calculate model performance. The combined probability file becomes the fixed input to notebook 18, keeping prediction generation separate from final evaluation.

In [ ]:
import os

# Limit BLAS/OpenMP thread counts to avoid runtime conflicts when XGBoost and PyTorch coexist on macOS.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
import torch

from catboost import CatBoostClassifier
from rtdl_revisiting_models import FTTransformer
from torch.utils.data import DataLoader, TensorDataset

# Explicit guard showing that this is the first notebook allowed to access the locked test set.
UNLOCK_FINAL_TEST = True

if not UNLOCK_FINAL_TEST:
    raise RuntimeError(
        "Final test is still locked. "
        "Run notebooks 11-16 first, check their saved artifacts, "
        "then set UNLOCK_FINAL_TEST = True in this cell."
    )


## Locate the project and check that every frozen model artifact exists

In [2]:
# Locate the project and verify that all six frozen model files exist before any test prediction is generated.

PROJECT_ROOT = Path.cwd()
for parent in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
    candidate = parent / "Processed_Dataset" / "diabetic_data_cleaned_stage1.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_ROOT = parent
        break
else:
    raise FileNotFoundError(
        "Could not find Processed_Dataset/diabetic_data_cleaned_stage1.csv"
    )

FINAL_EVALUATION_DIR = PROJECT_ROOT / "Final_Evaluation"
SPLIT_DIR = FINAL_EVALUATION_DIR / "Data_Splits"
ARTIFACT_ROOT = FINAL_EVALUATION_DIR / "Model_Artifacts"
RESULTS_DIR = FINAL_EVALUATION_DIR / "Results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

artifact_paths = {
    "logistic_regression": ARTIFACT_ROOT / "logistic_regression" / "final_model.joblib",
    "decision_tree": ARTIFACT_ROOT / "decision_tree" / "final_model.joblib",
    "random_forest": ARTIFACT_ROOT / "random_forest" / "final_model.joblib",
    "xgboost": ARTIFACT_ROOT / "xgboost" / "final_model.joblib",
    "catboost": ARTIFACT_ROOT / "catboost" / "final_model.cbm",
    "ft_transformer": ARTIFACT_ROOT / "ft_transformer" / "final_model_state_dict.pt",
}

# Failing here prevents a partial comparison where some models are missing.
missing = [str(path) for path in artifact_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Some frozen model artifacts are missing:\n" + "\n".join(missing)
    )

print("All six frozen model artifacts found.")
print("Results directory:", RESULTS_DIR)


All six frozen model artifacts found.
Results directory: /Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Results


## Load only the locked final-test rows

In [3]:
# Load the cleaned dataset, then select only the row positions frozen by notebook 10.

df = pd.read_csv(DATA_PATH)

target_col = "readmitted_30"

categorical_features = [
    "gender",
    "race_group",
    "age_group",
    "admission_source_group",
    "discharge_group",
    "medical_specialty_group",
    "primary_diagnosis",
    "hba1c_group",
    "max_glu_serum",
    "diabetesMed",
]

numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

model_features = categorical_features + numeric_features

missing_features = [c for c in model_features + [target_col] if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing required columns: {missing_features}")

X = df[model_features].copy()
y = df[target_col].astype(int).copy()

# These row positions are the shared lockbox used by every model.
test_idx = (
    pd.read_csv(SPLIT_DIR / "final_test_rows_LOCKED.csv")["row_position"]
    .astype(int)
    .to_numpy()
)

# All six models receive the same patients in the same row order.
X_test = X.iloc[test_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Locked test rows loaded:", len(X_test))
print("Positive count:", int(y_test.sum()))
print("Positive rate:", float(y_test.mean()))


Locked test rows loaded: 13998
Positive count: 1257
Positive rate: 0.08979854264894985


## Logistic Regression, Decision Tree, Random Forest, and XGBoost

In [4]:
# Load each frozen sklearn/XGBoost model and produce class-1 readmission probabilities.

# Logistic Regression
lr_dir = ARTIFACT_ROOT / "logistic_regression"
lr_model = joblib.load(lr_dir / "final_model.joblib")

with open(lr_dir / "model_variant.json") as f:
    lr_variant = json.load(f)["variant"]

# The selected Logistic Regression may require the engineered HbA1c × diagnosis feature.
if lr_variant == "interaction":
    X_lr_test = X_test.copy()
    X_lr_test["hba1c_x_primary_diagnosis"] = (
        X_lr_test["hba1c_group"].astype("string").fillna("Missing")
        + " | "
        + X_lr_test["primary_diagnosis"].astype("string").fillna("Missing")
    )
else:
    X_lr_test = X_test

# [:, 1] keeps the probability assigned to the positive/readmitted class.
lr_probability = lr_model.predict_proba(X_lr_test)[:, 1]

# Decision Tree
dt_model = joblib.load(ARTIFACT_ROOT / "decision_tree" / "final_model.joblib")
dt_probability = dt_model.predict_proba(X_test)[:, 1]

# Random Forest
rf_model = joblib.load(ARTIFACT_ROOT / "random_forest" / "final_model.joblib")
rf_probability = rf_model.predict_proba(X_test)[:, 1]

# XGBoost
xgb_model = joblib.load(ARTIFACT_ROOT / "xgboost" / "final_model.joblib")
xgb_probability = xgb_model.predict_proba(X_test)[:, 1]

print("Loaded and scored LR, DT, RF, and XGBoost.")


Loaded and scored LR, DT, RF, and XGBoost.


## CatBoost

In [5]:

def prepare_catboost_dataframe(X_data):
    """Recreate the input format used when the frozen CatBoost model was trained.

    Categorical variables remain strings for CatBoost's native categorical handling,
    while numerical variables are coerced to numeric values.
    """
    X_prepared = X_data.copy()

    for feature in categorical_features:
        values = X_prepared[feature].astype("object")
        values = values.where(values.notna(), "Missing")
        X_prepared[feature] = values.astype(str)

    for feature in numeric_features:
        X_prepared[feature] = pd.to_numeric(
            X_prepared[feature],
            errors="coerce",
        )

    return X_prepared

# Reconstruct an empty CatBoost estimator, load its saved state, then score the shared test rows.
cat_model = CatBoostClassifier()
cat_model.load_model(str(ARTIFACT_ROOT / "catboost" / "final_model.cbm"))

X_cat_test = prepare_catboost_dataframe(X_test)
cat_probability = cat_model.predict_proba(X_cat_test)[:, 1]

print("Loaded and scored CatBoost.")


Loaded and scored CatBoost.


## FT-Transformer

In [6]:

RANDOM_SEED = 42

# Use CUDA, Apple MPS, or CPU in that order for inference.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("FT-Transformer inference device:", DEVICE)

def build_ft_transformer(n_cont_features, cat_cardinalities, architecture):
    """Rebuild the exact FT-Transformer architecture selected during tuning."""
    model = FTTransformer(
        n_cont_features=n_cont_features,
        cat_cardinalities=cat_cardinalities,
        # Binary classification uses one output logit.
        d_out=1,
        n_blocks=int(architecture["n_blocks"]),
        d_block=int(architecture["d_block"]),
        attention_n_heads=int(architecture["attention_n_heads"]),
        attention_dropout=float(architecture["attention_dropout"]),
        ffn_d_hidden=None,
        ffn_d_hidden_multiplier=float(architecture["ffn_d_hidden_multiplier"]),
        ffn_dropout=float(architecture["ffn_dropout"]),
        residual_dropout=float(architecture["residual_dropout"]),
    )
    return model.to(DEVICE)


def make_prediction_loader(x_cont, x_cat, batch_size):
    """Package continuous and categorical test arrays into deterministic inference batches."""
    dataset = TensorDataset(
        torch.tensor(x_cont, dtype=torch.float32),
        torch.tensor(x_cat, dtype=torch.long),
    )
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=False,
        drop_last=False,
        pin_memory=(DEVICE.type == "cuda"),
    )


@torch.inference_mode()
def predict_probabilities(model, x_cont, x_cat, batch_size):
    """Run FT-Transformer inference and convert output logits to readmission probabilities."""
    model.eval()
    loader = make_prediction_loader(x_cont, x_cat, batch_size)
    probabilities = []

    for batch_cont, batch_cat in loader:
        batch_cont = batch_cont.to(DEVICE)
        batch_cat = batch_cat.to(DEVICE)

        logits = model(batch_cont, batch_cat).squeeze(-1)
        probabilities.append(
# Sigmoid maps the model's raw logit to a probability between 0 and 1.
            torch.sigmoid(logits).detach().cpu().numpy()
        )

    return np.concatenate(probabilities)


FT-Transformer inference device: mps


In [7]:
# Recreate exactly the categorical mappings and numerical transformations learned during FT-Transformer fitting.

ftt_dir = ARTIFACT_ROOT / "ft_transformer"

with open(ftt_dir / "selected_architecture.json") as f:
    ftt_architecture = json.load(f)

with open(ftt_dir / "selected_training_config.json") as f:
    ftt_training_config = json.load(f)

with open(ftt_dir / "category_mappings.json") as f:
    category_maps = json.load(f)

numeric_preprocessing = pd.read_csv(
    ftt_dir / "numeric_preprocessing.csv"
).set_index("feature")

# Map categories with the saved training-only lookup tables; unseen test categories use reserved ID 0.
# Categorical transformation. 0 is reserved for unseen categories.
cat_columns = []
cat_cardinalities = []

for feature in categorical_features:
    mapping = category_maps[feature]
    values = (
        X_test[feature]
        .astype("object")
        .where(X_test[feature].notna(), "Missing")
        .astype(str)
    )

    encoded = (
        values.map(mapping)
        .fillna(0)
        .astype(np.int64)
        .to_numpy()
    )
    cat_columns.append(encoded)
    cat_cardinalities.append(len(mapping) + 1)

test_cat = np.column_stack(cat_columns).astype(np.int64)

# Reuse the saved model-training medians, means and scales without fitting anything on test data.
# Numeric transformation using the training-only medians and scaling parameters.
numeric_matrix = []

for feature in numeric_features:
    values = pd.to_numeric(X_test[feature], errors="coerce").to_numpy(dtype=float)

    median = float(numeric_preprocessing.loc[feature, "imputer_median"])
    mean = float(numeric_preprocessing.loc[feature, "scaler_mean"])
    scale = float(numeric_preprocessing.loc[feature, "scaler_scale"])

    values = np.where(np.isnan(values), median, values)

    if not np.isfinite(scale) or scale == 0:
        scale = 1.0

    values = (values - mean) / scale
    numeric_matrix.append(values)

test_cont = np.column_stack(numeric_matrix).astype(np.float32)

# Build the selected architecture first, then restore the frozen neural-network weights.
ftt_model = build_ft_transformer(
    n_cont_features=test_cont.shape[1],
    cat_cardinalities=cat_cardinalities,
    architecture=ftt_architecture,
)

state_path = ftt_dir / "final_model_state_dict.pt"
try:
    state_dict = torch.load(
        state_path,
        map_location=DEVICE,
        weights_only=True,
    )
except TypeError:
    state_dict = torch.load(
        state_path,
        map_location=DEVICE,
    )

ftt_model.load_state_dict(state_dict)

# Score the same locked test patients and return one class-1 probability per row.
ftt_probability = predict_probabilities(
    model=ftt_model,
    x_cont=test_cont,
    x_cat=test_cat,
    batch_size=int(ftt_training_config["batch_size"]),
)

print("Loaded and scored FT-Transformer.")


Loaded and scored FT-Transformer.


## Save one combined prediction file

In [8]:
# Combine truth labels and all six model probabilities by shared row position.
# Keeping row-level predictions together is essential for the later paired bootstrap.

final_test_predictions = pd.DataFrame({
    "row_position": test_idx,
    "y_true": np.asarray(y_test, dtype=int),
    "logistic_regression": np.asarray(lr_probability, dtype=float),
    "decision_tree": np.asarray(dt_probability, dtype=float),
    "random_forest": np.asarray(rf_probability, dtype=float),
    "xgboost": np.asarray(xgb_probability, dtype=float),
    "catboost": np.asarray(cat_probability, dtype=float),
    "ft_transformer": np.asarray(ftt_probability, dtype=float),
})

probability_columns = [
    "logistic_regression",
    "decision_tree",
    "random_forest",
    "xgboost",
    "catboost",
    "ft_transformer",
]

# Sanity checks ensure every model produced one valid probability for every locked row.
assert len(final_test_predictions) == len(test_idx)
assert not final_test_predictions[probability_columns].isna().any().any()

for column in probability_columns:
    if not final_test_predictions[column].between(0, 1).all():
        raise ValueError(f"Invalid probability values in {column}")

OUTPUT_PATH = RESULTS_DIR / "final_test_predictions.csv"
final_test_predictions.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(final_test_predictions))


Saved: /Users/xiaohanmu/Desktop/School/UoB/Summer Individual Project/Processing_Pipeline/Pipeline_Selection/Final_Evaluation/Results/final_test_predictions.csv
Rows: 13998
Do not change or retune a model after this point. If you do, the final test is no longer a clean lockbox.
